In [1]:
import os
os.chdir(r"C:\Users\HP\OneDrive\Desktop\aiml-crash-pratham\phase2_mini_project")
print("Current folder:", os.getcwd())
print("Data files:", os.listdir("data/"))

Current folder: C:\Users\HP\OneDrive\Desktop\aiml-crash-pratham\phase2_mini_project
Data files: ['notebooks', 'olist_customers_dataset.csv', 'olist_orders_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_products_dataset.csv']


In [2]:
import sqlite3
import pandas as pd

# Load cleaned orders (from task 1)
orders   = pd.read_csv("data/olist_orders_dataset.csv")
payments = pd.read_csv("data/olist_order_payments_dataset.csv")
customers= pd.read_csv("data/olist_customers_dataset.csv")
items    = pd.read_csv("data/olist_order_items_dataset.csv")

# Push to SQLite
conn = sqlite3.connect("olist.db")
orders.to_sql("orders",    conn, if_exists="replace", index=False)
payments.to_sql("payments", conn, if_exists="replace", index=False)
customers.to_sql("customers",conn, if_exists="replace", index=False)
items.to_sql("items",      conn, if_exists="replace", index=False)

# ── 10 SQL Queries ─────────────────────────────────────

# Q1: Total orders
pd.read_sql("SELECT COUNT(*) as total_orders FROM orders", conn)

# Q2: Orders by status
pd.read_sql("SELECT order_status, COUNT(*) as cnt FROM orders GROUP BY order_status ORDER BY cnt DESC", conn)

# Q3: Average payment value
pd.read_sql("SELECT AVG(payment_value) as avg_payment FROM payments", conn)

# Q4: Top 5 states by orders (JOIN)
pd.read_sql("""
SELECT c.customer_state, COUNT(o.order_id) as total
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
GROUP BY c.customer_state
ORDER BY total DESC
LIMIT 5
""", conn)

# Q5: Payment type breakdown
pd.read_sql("""
SELECT payment_type, COUNT(*) as cnt, AVG(payment_value) as avg_val
FROM payments
GROUP BY payment_type
ORDER BY cnt DESC
""", conn)

# Q6: Orders per month (using substr on date string)
pd.read_sql("""
SELECT substr(order_purchase_timestamp,1,7) as month,
       COUNT(*) as orders
FROM orders
GROUP BY month
ORDER BY month
""", conn)

# Q7: Delivered orders only
pd.read_sql("""
SELECT COUNT(*) as delivered
FROM orders
WHERE order_status = 'delivered'
""", conn)

# Q8: High value orders > 500
pd.read_sql("""
SELECT o.order_id, p.payment_value
FROM orders o
JOIN payments p ON o.order_id = p.order_id
WHERE p.payment_value > 500
ORDER BY p.payment_value DESC
""", conn)

# Q9: Items per order (AVG)
pd.read_sql("""
SELECT AVG(item_count) as avg_items_per_order FROM (
  SELECT order_id, COUNT(*) as item_count
  FROM items
  GROUP BY order_id
)
""", conn)  # This is your SUBQUERY ✅

# Q10: Orders WHERE estimated delivery was in Q4
pd.read_sql("""
SELECT COUNT(*) as q4_orders
FROM orders
WHERE substr(order_estimated_delivery_date,6,2) IN ('10','11','12')
""", conn)

,q4_orders
0,16556
